# Vanilla SFT: Qwen-VL Next-Action Prediction

This notebook trains a vision-language model with supervised fine-tuning (SFT) to predict the next web action from HTML + screenshot image pairs.

Target model: set `MODEL_ID` below to your Qwen3-VL-8B checkpoint path or hub ID.

In [4]:
%pip install gdown datasets

  Using cached gdown-6.0.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-6.0.0-py3-none-any.whl (18 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.8 MB/s  0:00:00
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 188.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 17

In [1]:
import os
import re
import gdown

LOCAL = os.environ["LOCAL"]
os.makedirs(LOCAL, exist_ok=True)

drive_url = "https://drive.google.com/file/d/1leHOcLv631qfOniHsI409CW1aC48cEt8/view?usp=sharing"

def normalize_drive_download_url(url: str) -> str:
    m = re.search(r"/d/([a-zA-Z0-9_-]+)", url)
    if not m:
        m = re.search(r"[?&]id=([a-zA-Z0-9_-]+)", url)
    if m:
        return f"https://drive.google.com/uc?id={m.group(1)}"
    return url

zip_path = os.path.join(LOCAL, "images.zip")
extract_path = os.path.join(LOCAL, "project_data")
gdown.download(normalize_drive_download_url(drive_url), zip_path, quiet=False)

!unzip -qo $zip_path -d $extract_path
!rm -rf $zip_path

print("Downloaded and extracted to:", extract_path)

Downloading...
From (original): https://drive.google.com/uc?id=1leHOcLv631qfOniHsI409CW1aC48cEt8
From (redirected): https://drive.google.com/uc?id=1leHOcLv631qfOniHsI409CW1aC48cEt8&confirm=t&uuid=e25b4ce6-4769-4af8-b49d-58d370312a70
To: /local/images.zip
100%|████████████████████████████████████████████████████████████████████████████████| 5.22G/5.22G [00:52<00:00, 99.5MB/s]


error: invalid zip file with overlapped components (possible zip bomb)
 To unzip the file anyway, rerun the command with UNZIP_DISABLE_ZIPBOMB_DETECTION=TRUE environmnent variable
Downloaded and extracted to: /local/project_data


In [2]:
!tree -L 2 $LOCAL/project_data

/local/project_data
├── __MACOSX
│   └── screenshots_matched_6445
└── screenshots_matched_6445
    ├── 1004.jpg
    ├── 1005.jpg
    ├── 1010.jpg
    ├── 1011.jpg
    ├── 1018.jpg
    ├── 101.jpg
    ├── 1020.jpg
    ├── 1021.jpg
    ├── 1023.jpg
    ├── 1024.jpg
    ├── 1025.jpg
    ├── 1026.jpg
    ├── 1027.jpg
    ├── 102.jpg
    ├── 1030.jpg
    ├── 1032.jpg
    ├── 1033.jpg
    ├── 1034.jpg
    ├── 1035.jpg
    ├── 1036.jpg
    ├── 1037.jpg
    ├── 1038.jpg
    ├── 1039.jpg
    ├── 103.jpg
    ├── 1041.jpg
    ├── 1042.jpg
    ├── 1044.jpg
    ├── 1045.jpg
    ├── 1046.jpg
    ├── 1047.jpg
    ├── 1048.jpg
    ├── 1049.jpg
    ├── 104.jpg
    ├── 1050.jpg
    ├── 1051.jpg
    ├── 1052.jpg
    ├── 1053.jpg
    ├── 1054.jpg
    ├── 1055.jpg
    ├── 1057.jpg
    ├── 1058.jpg
    ├── 1059.jpg
    ├── 105.jpg
    ├── 1061.jpg
    ├── 1062.jpg
    ├── 1063.jpg
    ├── 1064.jpg
    ├── 1065.jpg
    ├── 1066.jpg
    ├── 1067.jpg
    ├── 1068.jpg
    ├── 1069.jpg
    ├── 106.jpg
    ├── 10

In [3]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.6.0+cu124
CUDA available: True


In [2]:
import os, shutil, torch, subprocess
print("host:", os.uname().nodename)
print("slurm job:", os.environ.get("SLURM_JOB_ID"))
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))
print("nvidia-smi path:", shutil.which("nvidia-smi"))
if shutil.which("nvidia-smi"):
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True))

host: w003.ib.bridges2.psc.edu
slurm job: 40055144
torch: 2.6.0+cu124
cuda available: True
cuda devices: 1
gpu name: NVIDIA H100 80GB HBM3
nvidia-smi path: /usr/bin/nvidia-smi
GPU 0: NVIDIA H100 80GB HBM3 (UUID: GPU-3dd92a62-ffe9-0727-4a7d-30546790351c)



In [1]:
import os
import json
import random
import subprocess
from dataclasses import dataclass

import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    print('GPU name:', torch.cuda.get_device_name(0))

try:
    print('\n=== nvidia-smi ===')
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as e:
    print('nvidia-smi unavailable:', e)

Torch: 2.5.1+cu124
CUDA available: True
GPU count: 1
GPU name: NVIDIA H100 80GB HBM3

=== nvidia-smi ===
Sun Apr 19 11:47:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:C2:00.0 Off |                    0 |
| N/A   23C    P0             71W /  700W |       4MiB /  81559MiB |      0%      Default |
|                                  

## 1) Install/Import Dependencies

If needed, uncomment the pip cell and run once in your environment.

In [3]:
from PIL import Image
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)
from pathlib import Path
import gdown
import zipfile

/ocean/projects/cis260137p/abhyanka/.project/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Configure Training

This notebook expects two inputs:
- a Google Drive link to a zipped image archive, which is downloaded with `gdown` and extracted locally
- a JSON file that already contains the corresponding HTML and action data

Expected JSON records should include:
- `image`: image filename or relative path inside the extracted archive
- `html`: page HTML string (or a compacted summary)
- `action`: target next action text (recommend a deterministic JSON action format)

In [6]:
from dataclasses import dataclass
import os

@dataclass
class CFG:
    model_id = "Qwen/Qwen3-VL-8B-Thinking"
    hf_token = None
    hf_cache_dir = "./.hf_cache"
    use_flash_attn = True
    use_quantization = False
    image_root = os.path.join(os.environ.get("LOCAL", "."), "project_data")
    jsonl_path = "./vanilla_matched_6445.jsonl"
    output_dir = "./outputs/qwen-vl-next-action-sft"
    max_length = 2048

    epochs = 5
    lr = 1e-5
    train_bs = 1
    eval_bs = 1
    grad_accum = 4
    warmup_ratio = 0.03
    weight_decay = 0.01
    log_steps = 10
    save_steps = 200

    use_bf16 = True
    grad_checkpointing = True
    dataloader_num_workers = 0
    cache_clear_steps = 50
    max_prompt_chars = 800
    image_max_side = 320

    use_lora = True
    lora_r = 16
    lora_alpha = 32
    lora_dropout = 0.05
    lora_targets = ("q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj")

cfg = CFG()
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(os.path.dirname(cfg.jsonl_path), exist_ok=True)
os.makedirs(cfg.hf_cache_dir, exist_ok=True)
cfg

CFG()

## 3) Run `runner.py`

The notebook now delegates training and evaluation to the modular pipeline in `dataloading.py`, `train.py`, and `runner.py`.

In [ ]:
import importlib
import runner

runner = importlib.reload(runner)
runner.run_training(cfg)

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.
Dataset({
    features: ['image', 'system_prompt', 'prompt_text', 'target'],
    num_rows: 5343
})
Columns: ['image', 'system_prompt', 'prompt_text', 'target']
Train size: 5236
Eval size: 107
Memory settings: {'train_bs': 1, 'grad_accum': 4, 'effective_batch': 4, 'max_prompt_chars': 800, 'image_max_side': 320, 'cache_clear_steps': 50}
Loading model...
Trying model_id=Qwen/Qwen3-VL-8B-Thinking


Fetching 4 files: 100%|██████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 66313.11it/s]


Failed with flash_attention_2: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package for FlashAttention2 doesn't seem to be installed.
Falling back to sdpa...


Loading weights: 100%|█████████████████████████████████████████████████████████████████| 750/750 [00:06<00:00, 123.33it/s]


Model loaded with attn_implementation=sdpa


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


Step,Training Loss,Validation Loss
200,0.290169,0.328616
400,0.325110,0.308711
600,0.212068,0.240952
800,0.218061,0.215397
